# Chapter 26 — Message Types

**Companion to *Pi Agents*** · [`Pi Agents` chapter 26](https://programmer.ie/books/pi/26-chapter/)

| | |
|---|---|
| Chapter | `26-chapter.md` · weight 26 · `/books/pi/26-chapter/` |
| Notebook | `notebooks/pi/26-chapter.ipynb` |
| Pin | `@earendil-works/pi-*` **1.0.4** (`examples/package.json`) |
| Ground truth | `docs/*.md` and `dist/*.d.ts` **at the pin**, under `examples/node_modules/` |

> **Where the authority lives.** Every claim below is checked against the
> packages installed under `examples/node_modules/@earendil-works/`, not against
> a globally installed Pi and not from memory. When prose and an exported
> declaration disagree, the declaration wins.

## The question

Do the types printed in the chapter match the exported declarations — and does
`SystemMessage.replace` exist at this pin?

The chapter's point: the message types are a **contract**, and the contract is the
exported declaration, not the prose. When `message-types.md` describes a
`SystemMessage.replace` that the 1.0.4 declaration does not have, the declaration
wins and the discrepancy is recorded.


## What this notebook establishes

- The chapter's hand-written declarations are **assignable to and from** the exported types, in both directions, checked at compile time.
- The **key sets** match too — assignability alone tolerates a missing optional field, so the key sets are compared separately.
- At 1.0.4, the exported `SystemMessage` has **no `replace` member**, and no shipped JavaScript reads one.
- The four roles the chapter declares (`system`, `user`, `assistant`, `toolResult`) are the four in the exported union.

## What this notebook does **not** establish

- **Not a runtime claim.** This notebook checks **declarations and static text**; it runs no session and asserts no behaviour.
- **The prose/declaration mismatch on `replace`** is a recorded discrepancy, not a bug report.
- **No provider mapping is described** — the chapter does not claim how a message becomes a provider payload.
- **The declaration wins** where the two disagree; this notebook's job is to show where they do.


## Setup

This notebook runs the book's own examples. **Pi is Node/TypeScript**; Python
only presents, orchestrates and checks. There is no Python re-implementation
of Pi's loop, context loader, session projection, permissions or protocol
anywhere in this set — where a mechanism is demonstrated, the code that runs is
the code in `examples/`.

* **Execution mode:** **Inspect** — compile-time check (`tsc` over `assert-equal.ts`) plus a static read of the 1.0.4 declaration and the shipped JavaScript.
* **Evidence scope:** `declarations`. Nothing here runs a model or a session.
* **Credentials:** none. The model is a scripted stand-in (`faux`) supplied by
  the book's harness, so nothing here reaches the network, spends money, or
  says anything about how a real model behaves.


In [1]:
# --- Locate the shared support module and the pinned packages -----------------
# Two documented markers, both overridable:
#   PIN_REPO       -> the checkout of the pi repository
#   PIN_NOTEBOOKS  -> this notebook directory
# Because both searches walk upwards from the working directory, this cell
# works whether you launched Jupyter from notebooks/pi/ or from the repo root.
import sys
from pathlib import Path


def _find_support() -> Path:
    for c in (Path.cwd(), *Path.cwd().parents):
        if (c / "pinb.py").is_file():
            return c
        nested = c / "notebooks" / "pi"
        if (nested / "pinb.py").is_file():
            return nested
    raise SystemExit(
        "Could not find notebooks/pi/pinb.py from " + str(Path.cwd()) + ".\n"
        "Open this notebook from notebooks/pi/, or set PIN_NOTEBOOKS."
    )


sys.path.insert(0, str(_find_support()))
import pinb  # noqa: E402  (a local module, deliberately after the path fix)

pinb.self_check()


{'repo_root': 'resolved',
 'notebook_dir': 'resolved',
 'pin_matches_manuscript': True,
 'node': 'v25.8.2',
 'python': '3.13.15',
 'claim_ledger': True,
 'credentials_required': False}

In [2]:
pinb.show_environment("This notebook's runtime")

print()
print("Repository marker: an ancestor directory holding both")
print("  content/books/pi/                          (the manuscript)")
print("  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)")
print("Every path below is shown relative to that repository; no absolute")
print("path is printed, so the saved output is portable.")
print()
print(pinb.md_table(
    ["resource", "repository-relative path"],
    [["manuscript", pinb.rel(pinb.chapter_file(26))],
     ["examples tree", "examples/"],
     ["claim ledger", "examples/evidence.json"],
     ["pinned docs", "examples/node_modules/@earendil-works/pi-coding-agent/docs/"],
     ["pinned declarations", "examples/node_modules/@earendil-works/*/dist/"],
     ["support module", "notebooks/pi/pinb.py"]],
))


== This notebook's runtime ==
component  value
-------------------------------  ----------------------------------------------------
manuscript pin (AGENTS.md)  1.0.4
@earendil-works/pi-ai  1.0.4
@earendil-works/pi-agent-core  1.0.4
@earendil-works/pi-coding-agent  1.0.4
node  v25.8.2
python  3.13.15
platform  win32
examples tree  installed
model used  faux (scripted) - no network, no credential, no cost

Repository marker: an ancestor directory holding both
  content/books/pi/                          (the manuscript)
  examples/node_modules/@earendil-works/pi-coding-agent   (the pin)
Every path below is shown relative to that repository; no absolute
path is printed, so the saved output is portable.

| resource | repository-relative path |
|---|---|
| manuscript | content/books/pi/26-chapter.md |
| examples tree | examples/ |
| claim ledger | examples/evidence.json |
| pinned docs | examples/node_modules/@earendil-works/pi-coding-agent/docs/ |
| pinned declarations | examples/node_mod

## Baseline: the declaration at the pin

The exported `SystemMessage`, read directly from
`examples/node_modules/@earendil-works/pi-ai/dist/types.d.ts`. This is the
authority the chapter is checked against.

In [3]:
TYPES = pinb.driver_source("ch26-message-types.ts")
types = pinb.driver(TYPES, label="ch26-message-types", timeout=120)
print("SystemMessage at 1.0.4:")
print("  " + types["systemMessageDeclaration"][:400])
print()
print("Four roles in the chapter's declaration:", ", ".join(types["declaredRoles"]))

SystemMessage at 1.0.4:
  export interface SystemMessage { role: "system"; /** Instruction text. On the leading message this is the base prompt; later, additional instructions. */ content: string | TextContent[]; /** * Named, ordered prompt sections rendered verbatim after `content`. The leading message * declares them; later messages replace sections by name, and `null` removes one. Keep * each section self-delimiting (a 

Four roles in the chapter's declaration: system, user, assistant, toolResult


In [4]:
pinb.show_checks([
    pinb.check("the four base roles are declared",
               types["declaredRoles"] == ["system", "user", "assistant", "toolResult"],
               ", ".join(types["declaredRoles"])),
    pinb.check("the declaration still has content, sections, toolsAdded, toolsRemoved at this pin",
               all(k in types["systemMessageDeclaration"] for k in ["content", "sections", "toolsAdded", "toolsRemoved"]),
               "all four fields present"),
])

== Assertions ==
  assertion  observed
----  --------------------------------------------------------------------------  -----------------------------------
PASS  the four base roles are declared  system, user, assistant, toolResult
PASS  the declaration still has content, sections, toolsAdded, toolsRemoved at …  all four fields present

2/2 assertions held.


## The compile-time check

`ch26-message-types/assert-equal.ts` compares the chapter's declarations with the
exported types, in both directions, and separately compares the key sets. If a
field drifts on either side, `tsc` fails. Running the project typecheck is
running the chapter's declaration claim.

In [5]:
import subprocess
r = pinb.run_node(["node_modules/typescript/bin/tsc", "-p", "."], cwd=pinb.EXAMPLES, timeout=180)
print("$ tsc -p examples/")
print(f"  exit={r.status}")
if r.stdout.strip():
    print(r.stdout.strip()[:500])
if r.stderr.strip():
    print(r.stderr.strip()[:500])
pinb.show_checks([
    pinb.check("the chapter declarations compile against the exported types",
               r.status == 0, "tsc exit 0"),
])

$ tsc -p examples/
  exit=0
== Assertions ==
  assertion  observed
----  -----------------------------------------------------------  ----------
PASS  the chapter declarations compile against the exported types  tsc exit 0

1/1 assertions held.


## Contrasting case: the recorded discrepancy

`message-types.md` describes `SystemMessage.replace`. The 1.0.4 declaration does
not have one, and no shipped JavaScript reads one. The declaration wins: a
notebook or a chapter that used `replace` would fail to compile.

This is the book's drift protocol in miniature — a documented member that the
declaration never had. It is **recorded**, not smoothed over.

In [6]:
print("Does the 1.0.4 SystemMessage declaration have a `replace` member?")
print("  " + ("yes" if types["declarationHasReplace"] else "no"))
print()
print(f"Shipped JavaScript files scanned: {types['jsFilesScanned']}")
print("Files that read a SystemMessage.replace:", types["jsReadsReplace"] or "(none)")
print()
pinb.show_checks([
    pinb.check("the 1.0.4 declaration has NO `replace` member",
               not types["declarationHasReplace"], "absent from the declaration"),
    pinb.check("no shipped JavaScript reads a SystemMessage.replace",
               len(types["jsReadsReplace"]) == 0, "no reader found"),
])

Does the 1.0.4 SystemMessage declaration have a `replace` member?
  no

Shipped JavaScript files scanned: 24
Files that read a SystemMessage.replace: (none)

== Assertions ==
  assertion  observed
----  ---------------------------------------------------  ---------------------------
PASS  the 1.0.4 declaration has NO `replace` member  absent from the declaration
PASS  no shipped JavaScript reads a SystemMessage.replace  no reader found

2/2 assertions held.


## Your turn: inspect, then decide

**Predict first.** The chapter's `assert-equal.ts` compares both assignability and
key sets. Which would catch an **optional** field the declaration gained but the
chapter did not? (Assignability would not; the key-set comparison would.)

**Then check one thing.** Open
`examples/node_modules/@earendil-works/pi-ai/dist/types.d.ts` and find
`AssistantMessage`. Does it declare `stopReason`? What are the allowed values?

**Predict the boundary.** If a future Pi release *adds* `SystemMessage.replace`,
what breaks first — this notebook's static check, the chapter's `assert-equal.ts`,
or the prose that documents it?

In [7]:
print("Predictions:")
print("  1. Key-set comparison catches an added optional field; assignability does not.")
print("  2. AssistantMessage declares stopReason with the union of stop reasons.")
print("  3. A new member breaks the static check first (this notebook), then tsc, then the prose.")
print()
print("Observed above:")
print(f"  declaration has replace: {types['declarationHasReplace']}")
print(f"  shipped JS readers: {len(types['jsReadsReplace'])}")
print(f"  roles declared: {types['declaredRoles']}")
assert not types["declarationHasReplace"]
assert len(types["jsReadsReplace"]) == 0
print()
print("held: the 1.0.4 declaration has no SystemMessage.replace; the prose describes one; the declaration wins")

Predictions:
  1. Key-set comparison catches an added optional field; assignability does not.
  2. AssistantMessage declares stopReason with the union of stop reasons.
  3. A new member breaks the static check first (this notebook), then tsc, then the prose.

Observed above:
  declaration has replace: False
  shipped JS readers: 0
  roles declared: ['system', 'user', 'assistant', 'toolResult']

held: the 1.0.4 declaration has no SystemMessage.replace; the prose describes one; the declaration wins


## Interpretation

The message types are checked two ways, and each catches something the other
misses:

| Check | What it catches | What it misses |
|---|---|---|
| Assignability (`Same<A, B>`) | A type that is structurally incompatible | An **optional** field missing on one side |
| Key sets (`SameKeys<A, B>`) | Any field the two disagree about | Nothing at the type level |
| Static read of `.d.ts` | A member the prose describes but the declaration lacks | Whether the shipped code reads it |
| Static read of `.js` | Whether the shipped code actually uses a member | Behaviour |

The practical rules:
1. **The declaration wins.** When prose and `.d.ts` disagree, the book follows the declaration.
2. **Check both directions, and the key sets.** Assignability alone tolerates a missing optional field.
3. **A discrepancy is recorded, not smoothed over.** `SystemMessage.replace` is the book's worked example.
4. **Compile-time is evidence for a type claim.** `tsc` passing on `assert-equal.ts` is the chapter's declaration evidence.

## Sources, execution mode and limitations

**Execution mode:** **Inspect** — compile-time check (`tsc` over `assert-equal.ts`) plus a static read of the 1.0.4 declaration and the shipped JavaScript.

**Evidence scope:** `declarations`. Nothing here runs a model or a session.

### What was read or run

- **Ran** `tsc -p examples/` — the chapter's `assert-equal.ts` compiled against the pinned declarations.
- **Read** `@earendil-works/pi-ai/dist/types.d.ts` for the exported `SystemMessage`.
- **Read** `ch26-message-types/declared.ts` and `assert-equal.ts`.
- **Read** `examples/evidence.json`, chapter 26 row (1 claim, `declarations`).

### Limitations

- **Not a runtime claim.** This notebook checks declarations and static text only.
- **The prose/declaration mismatch on `replace`** is a recorded discrepancy, not a bug report.
- **No provider mapping is described** — the chapter does not claim how a message becomes a provider payload.

Every technical statement in this notebook is about Pi **1.0.4**. A later
release can change any of it; the book's drift protocol in `AGENTS.md` is the
procedure for finding out whether it did.
